In [1]:
import sys
import logging
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# 1. Logger Configuration
# -------------------------------------------------------------------------
# Configure a non-root logger at module level
logger = logging.getLogger(__name__)


def setup_logging(log_filename: str = "pipeline.log", debug_mode: bool = False) -> None:
    """
    Configures logging handlers for feature engineering module.
    """
    logger.setLevel(logging.DEBUG)  # Base logger captures DEBUG level

    if logger.hasHandlers():
        logger.handlers.clear()

    formatter = logging.Formatter(
        fmt="%(asctime)s — %(levelname)s — %(name)s — %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    # Console Handler (INFO by default, DEBUG if debug_mode=True)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.DEBUG if debug_mode else logging.INFO)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    # File Handler (Always records DEBUG messages to file)
    file_handler = logging.FileHandler(log_filename, mode='a', encoding='utf-8')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)


# -------------------------------------------------------------------------
# 2. Feature Engineering Logic
# -------------------------------------------------------------------------
def create_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Engineers temporal variables: hour, day of week, weekend flag, and cyclical sine/cosine encodings.
    """
    df = df.copy()
    
    # Basic calendar components
    df['hour'] = df['date_time'].dt.hour
    df['day_of_week'] = df['date_time'].dt.dayofweek  # 0 = Monday, 6 = Sunday
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Cyclical Encoding for 24-Hour Cycle
    # Captures continuous transition between 23:00 and 00:00
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    
    # Cyclical Encoding for 7-Day Week Cycle
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)

    logger.debug("Engineered time features: hour, day_of_week, is_weekend, hour_sin, hour_cos, day_sin, day_cos.")
    return df


def create_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Encodes categorical weather variables and derives domain-specific weather indicators.
    """
    df = df.copy()

    # One-Hot Encoding for primary weather category
    weather_dummies = pd.get_dummies(df['weather_main'], prefix='weather', drop_first=False).astype(int)
    df = pd.concat([df, weather_dummies], axis=1)

    # Derived Weather Indicators
    df['is_precipitating'] = ((df['rain_1h'] > 0) | (df['snow_1h'] > 0)).astype(int)
    df['temp_celsius'] = df['temp'] - 273.15
    df['is_freezing'] = (df['temp_celsius'] < 0).astype(int)

    logger.debug(f"Engineered weather features: One-hot encoded {weather_dummies.shape[1]} weather types, "
                 f"is_precipitating, temp_celsius, is_freezing.")
    return df


def create_numerical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Scales continuous variables (Min-Max Normalisation) to range [0, 1].
    """
    df = df.copy()

    # Scaled Temperature
    t_min, t_max = df['temp'].min(), df['temp'].max()
    df['temp_scaled'] = (df['temp'] - t_min) / (t_max - t_min)

    # Scaled Cloud Coverage
    c_min, c_max = df['clouds_all'].min(), df['clouds_all'].max()
    df['clouds_scaled'] = (df['clouds_all'] - c_min) / (c_max - c_min)

    # Intermediate calculation logging at DEBUG level
    logger.debug(f"Min-Max Normalisation stats — temp range: [{t_min:.2f}, {t_max:.2f}] K | "
                 f"clouds_all range: [{c_min:.2f}, {c_max:.2f}]%")

    return df


def create_traffic_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Categorises continuous traffic volume into data-driven congestion levels (Low, Moderate, High)
    using tertile thresholds (33.3rd and 66.6th percentiles).
    """
    df = df.copy()

    # Calculate tertile thresholds
    q33 = df['traffic_volume'].quantile(0.333)
    q66 = df['traffic_volume'].quantile(0.666)

    # Log intermediate threshold values at DEBUG level
    logger.debug(f"Intermediate Quantile Thresholds calculated for Congestion Categorisation — "
                 f"33.3rd percentile (Low/Moderate cutoff): {q33:.1f} vehicles/hr | "
                 f"66.6th percentile (Moderate/High cutoff): {q66:.1f} vehicles/hr")

    def categorize_congestion(vol: float) -> str:
        if vol <= q33:
            return 'Low'
        elif vol <= q66:
            return 'Moderate'
        else:
            return 'High'

    df['congestion_level'] = df['traffic_volume'].apply(categorize_congestion)
    return df


def engineer_features(input_filepath: str = "cleaned_traffic_data.csv",
                       output_filepath: str = "featured_traffic_data.csv",
                       debug_mode: bool = False) -> pd.DataFrame:
    """
    Orchestrates the feature engineering workflow.
    """
    setup_logging(log_filename="pipeline.log", debug_mode=debug_mode)
    logger.info("================ Starting Feature Engineering Phase ================")

    # Load cleaned data
    df = pd.read_csv(input_filepath)
    df['date_time'] = pd.to_datetime(df['date_time'])

    # Log initial dataset shape
    initial_rows, initial_cols = df.shape
    logger.info(f"Dataset shape BEFORE feature engineering: {initial_rows} rows x {initial_cols} columns")

    # Apply transformations
    df = create_time_features(df)
    df = create_weather_features(df)
    df = create_numerical_features(df)
    df = create_traffic_target(df)

    # Save output dataset
    df.to_csv(output_filepath, index=False)

    # Log final dataset shape
    final_rows, final_cols = df.shape
    logger.info(f"Dataset shape AFTER feature engineering: {final_rows} rows x {final_cols} columns")
    logger.info(f"Feature engineering completed successfully. Output saved to '{output_filepath}'.")
    logger.info("==================================================================")

    return df


if __name__ == "__main__":
    engineer_features(input_filepath="cleaned_traffic_data.csv", debug_mode=False)

2026-10-05 14:04:25 — INFO — __main__ — ================ Starting Feature Engineering Phase ================
2026-10-05 14:04:25 — INFO — __main__ — Dataset shape BEFORE feature engineering: 48187 rows x 9 columns
2026-10-05 14:04:26 — INFO — __main__ — Dataset shape AFTER feature engineering: 48187 rows x 33 columns
2026-10-05 14:04:26 — INFO — __main__ — Feature engineering completed successfully. Output saved to 'featured_traffic_data.csv'.
2026-10-05 14:04:26 — INFO — __main__ — ==================================================================
